# 🚦 Traffic Accidents — Complete Analysis
**Pipeline:** Data Loading → Cleaning → EDA & Visualization → Machine Learning → Prediction

Dataset: 209,306 traffic accident records with 24 features.

---
## 1. Setup & Data Loading

In [1]:
%pip install -q plotly

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
sns.set_theme(style='whitegrid', palette='muted')

print('All libraries loaded!')

In [ ]:
# Upload traffic_accidents.csv when prompted
from google.colab import files
uploaded = files.upload()
FILE_PATH = list(uploaded.keys())[0]

df = pd.read_csv(FILE_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

---
## 2. Data Cleaning & Preprocessing

In [ ]:
print('=== Dataset Info ===')
df.info()
print()
print('=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Drop nulls
df = df.dropna()
print(f'Shape after dropping nulls: {df.shape}')

# Parse crash_date and extract features
df['crash_date'] = pd.to_datetime(df['crash_date'])
df['date']  = df['crash_date'].dt.date
df['time']  = df['crash_date'].dt.time
df['year']  = df['crash_date'].dt.year
df = df.drop(columns=['crash_date'])

# Binary encode target: 0 = no injury, 1 = injury/severe
df['crash_type'] = df['crash_type'].apply(
    lambda x: 0 if x == 'NO INJURY / DRIVE AWAY' else 1
)

print('Target distribution:')
print(df['crash_type'].value_counts())
print()
df.head()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Temporal Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By hour
hour_counts = df['crash_hour'].value_counts().sort_index()
axes[0].bar(hour_counts.index, hour_counts.values, color='steelblue')
axes[0].set_title('Crashes by Hour of Day')
axes[0].set_xlabel('Hour (0–23)')
axes[0].set_ylabel('Count')

# By day of week
day_labels = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
day_counts = df['crash_day_of_week'].value_counts().sort_index()
axes[1].bar(day_labels, day_counts.values, color='coral')
axes[1].set_title('Crashes by Day of Week')
axes[1].set_xlabel('Day')

# By month
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df['crash_month'].value_counts().sort_index()
axes[2].bar(month_labels, month_counts.values, color='mediumseagreen')
axes[2].set_title('Crashes by Month')
axes[2].set_xlabel('Month')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Hour vs Day of Week
pivot = df.groupby(['crash_day_of_week', 'crash_hour']).size().unstack(fill_value=0)
pivot.index = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

plt.figure(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3)
plt.title('Crash Frequency — Hour of Day vs Day of Week')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.show()

### 3.2 Environmental Conditions

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(x='weather_condition', data=df, order=df['weather_condition'].value_counts().index[:10])
plt.xticks(rotation=45, ha='right')
plt.title('Accidents by Weather Condition (Top 10)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(x='weather_condition', hue='crash_type',
              data=df, order=df['weather_condition'].value_counts().index[:8])
plt.xticks(rotation=45, ha='right')
plt.title('Accident Severity by Weather Condition')
plt.legend(title='Crash Type', labels=['No Injury', 'Injury/Severe'])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x='lighting_condition', data=df,
              order=df['lighting_condition'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title('Accidents by Lighting Condition')
plt.tight_layout()
plt.show()

### 3.3 Road & Crash Type

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(x='first_crash_type', data=df,
              order=df['first_crash_type'].value_counts().index[:10])
plt.xticks(rotation=45, ha='right')
plt.title('Accidents by First Crash Type (Top 10)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.countplot(ax=axes[0], x='trafficway_type', data=df,
              order=df['trafficway_type'].value_counts().index[:8])
axes[0].set_title('Accidents by Road Type')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(ax=axes[1], x='trafficway_type', hue='crash_type', data=df,
              order=df['trafficway_type'].value_counts().index[:6])
axes[1].set_title('Accident Severity by Road Type')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Crash Type', labels=['No Injury', 'Injury/Severe'])

plt.tight_layout()
plt.show()

### 3.4 Injuries & Severity

In [ ]:
injury_cols = ['injuries_total', 'injuries_fatal', 'injuries_incapacitating',
               'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication']

print('=== Injury Totals Across All Crashes ===')
print(df[injury_cols].sum().astype(int))
print()
print('=== Average Per Crash ===')
print(df[injury_cols].mean().round(4))

In [ ]:
severity_counts = df['most_severe_injury'].value_counts()

plt.figure(figsize=(8, 6))
plt.pie(severity_counts.values, labels=severity_counts.index,
        autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Most Severe Injury')
plt.tight_layout()
plt.show()

### 3.5 Contributing Causes

In [ ]:
cause_counts = df['prim_contributory_cause'].value_counts().head(12)

plt.figure(figsize=(12, 6))
cause_counts[::-1].plot(kind='barh', color='tomato')
plt.title('Top 12 Primary Contributory Causes')
plt.xlabel('Number of Crashes')
plt.tight_layout()
plt.show()

### 3.6 Correlation Matrix

In [ ]:
num_cols = ['num_units', 'injuries_total', 'injuries_fatal',
            'injuries_incapacitating', 'injuries_non_incapacitating',
            'injuries_reported_not_evident', 'injuries_no_indication',
            'crash_hour', 'crash_day_of_week', 'crash_month', 'crash_type']

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 4. Machine Learning — Crash Severity Prediction

### 4.1 Feature Encoding & Train/Test Split

In [ ]:
features = [
    'weather_condition',
    'lighting_condition',
    'crash_hour',
    'trafficway_type',
    'first_crash_type'
]

# Use a separate dict of encoders so we can decode later
encoders = {}
df_model = df[features + ['crash_type']].copy()

categorical_features = ['weather_condition', 'lighting_condition', 'trafficway_type', 'first_crash_type']

for col in categorical_features:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le  # save each encoder separately

# Print encoding maps
for col, le in encoders.items():
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'\n{col}:')
    for k, v in mapping.items():
        print(f'  {v} = {k}')

In [ ]:
X = df_model[features]
y = df_model['crash_type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')

### 4.2 Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print('=== Decision Tree Results ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}')
print()
print(classification_report(y_test, y_pred_dt, target_names=['No Injury', 'Injury/Severe']))

In [ ]:
# Confusion matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['No Injury', 'Injury/Severe'])

plt.figure(figsize=(6, 5))
disp.plot(cmap='Blues', ax=plt.gca())
plt.title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.show()

### 4.3 Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('=== Random Forest Results ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['No Injury', 'Injury/Severe']))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['No Injury', 'Injury/Severe'])

plt.figure(figsize=(6, 5))
disp_rf.plot(cmap='Greens', ax=plt.gca())
plt.title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

### 4.4 Model Comparison

In [ ]:
dt_acc  = accuracy_score(y_test, y_pred_dt)
rf_acc  = accuracy_score(y_test, y_pred_rf)

models  = ['Decision Tree', 'Random Forest']
scores  = [dt_acc, rf_acc]
colors  = ['steelblue', 'mediumseagreen']

plt.figure(figsize=(7, 4))
bars = plt.bar(models, scores, color=colors, width=0.4)
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
             f'{score:.4f}', ha='center', va='top', fontsize=13, color='white', fontweight='bold')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()

winner = 'Random Forest' if rf_acc > dt_acc else 'Decision Tree'
print(f'Best model: {winner} ({max(scores):.4f})')

### 4.5 Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, model, name, color in [
    (axes[0], dt_model,  'Decision Tree',  'steelblue'),
    (axes[1], rf_model,  'Random Forest',  'mediumseagreen')
]:
    imp_df = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=True)

    ax.barh(imp_df['Feature'], imp_df['Importance'], color=color)
    ax.set_title(f'{name} — Feature Importance')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

---
## 5. Live Prediction (Using Random Forest)

In [ ]:
# Show encoding options for the user
print('===== ENCODING REFERENCE =====')
for col, le in encoders.items():
    print(f'\n{col}:')
    for i, cls in enumerate(le.classes_):
        print(f'  {i} = {cls}')

In [ ]:
# Enter values based on the encoding reference above
weather    = int(input('Weather condition (encoded number): '))
lighting   = int(input('Lighting condition (encoded number): '))
hour       = int(input('Crash hour (0–23): '))
road       = int(input('Trafficway type (encoded number): '))
crash_t    = int(input('First crash type (encoded number): '))

sample = [[weather, lighting, hour, road, crash_t]]
prediction = rf_model.predict(sample)
probability = rf_model.predict_proba(sample)[0]

print()
if prediction[0] == 1:
    print(f'⚠️  Prediction: SEVERE ACCIDENT')
else:
    print(f'✅  Prediction: MINOR / NO INJURY')

print(f'   Confidence — No Injury: {probability[0]:.1%} | Severe: {probability[1]:.1%}')

---
## 6. Key Insights Summary

In [ ]:
peak_hour  = df['crash_hour'].mode()[0]
peak_day   = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday'][df['crash_day_of_week'].mode()[0]]
top_weather = df['weather_condition'].mode()[0]
top_cause   = df['prim_contributory_cause'].mode()[0]
fatal_count = (df['injuries_fatal'] > 0).sum()
severe_pct  = df['crash_type'].mean() * 100

print('=' * 58)
print('         TRAFFIC ACCIDENTS — KEY INSIGHTS')
print('=' * 58)
print(f'  Total records         : {len(df):,}')
print(f'  Severe crash rate     : {severe_pct:.1f}%')
print(f'  Peak crash hour       : {peak_hour}:00')
print(f'  Peak crash day        : {peak_day}')
print(f'  Most common weather   : {top_weather}')
print(f'  Top cause             : {top_cause[:45]}')
print(f'  Crashes with fatality : {fatal_count:,}')
print(f'  Total injuries        : {int(df["injuries_total"].sum()):,}')
print(f'  Total fatalities      : {int(df["injuries_fatal"].sum()):,}')
print()
print('  Model Performance:')
print(f'    Decision Tree  : {dt_acc:.4f}')
print(f'    Random Forest  : {rf_acc:.4f}')
print('=' * 58)